# Filtering tables to obtain activity information only from the IP subject

The objective of this script is to filter the tables that collect information on the delivered tasks to obtain only those related to the IP course, and thus avoid having to do a join to concatenate the user with the task and another to join these tuples with only the tasks belonging to a course. 

When concatenating, we will only keep the useful attributes of each table.

## Configuration

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
import os

# Configuration
curso_ip = 8683
ruta_origen = "/home/carlos/Documentos/TFG/spark-workspace/data/raw"
ruta_destino = "/home/carlos/Documentos/TFG/spark-workspace/data/intermediate/concatenados"
os.makedirs(ruta_destino, exist_ok=True)

# Create Spark session
spark = SparkSession.builder \
    .appName("IP course data filtering") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/06/12 20:11:26 WARN Utils: Your hostname, carlos-Modern-15-A11SB, resolves to a loopback address: 127.0.1.1; using 172.17.0.1 instead (on interface docker0)
25/06/12 20:11:26 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/06/12 20:11:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/06/12 20:11:27 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/06/12 20:11:27 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


## Filtering of assignments and concatenation of deliveries and notes

After performing this operation, we will be left with a single table `assign_submission_grade_cmi` that will have information on all the assignments submitted by the ip students, along with the grade they obtained in them.

 It is key to understand that, since we will do an 'inner join' between the tasks of the subject, with the deliveries associated with them, only those tasks that have at least one delivery associated with them will be saved as a result of the union, which are those that are valid for the metrics that are intended to be calculated.

In [ ]:
# Read assignment and shipment data
from math import trunc


assign = spark.read.parquet(f"{ruta_origen}/assign_cmi.parquet")
submissions = spark.read.parquet(f"{ruta_origen}/assign_submission_cmi.parquet")


# Filter table to only have the ip tasks, and only have the necessary fields
assign_ip = assign.filter(col("course") == curso_ip).select(
    "id", "duedate", "allowsubmissionsfromdate", "name"
)
# We concatenate tasks with their deliveries in such a way that tasks that have no associated delivery will not be obtained.
submissions_ip = (
    submissions.join(assign_ip, submissions.assignment == assign_ip.id, "inner")
    .withColumnRenamed("timemodified", "timesubmitted")
    .drop("id")
)



number_of_submissions = submissions_ip.count()

print(f"Number of assignment submissions for the course {course_ip}: {number_of_submissions}")


# Concatenate submissions with grades, to have information on submissions and notes.
grades = spark.read.parquet(f"{ruta_origen}/assign_grades_cmi.parquet")
grades_filtered = grades.select("userid", "assignment", "grade")

assign_submission_grade_cmi = submissions_ip.join(
    grades_filtered, on=["userid", "assignment"], how="left"
).drop("status")

# We check the number of tuples after the union, to verify if the number of tuples increases,
# which could mean that there may be several notes associated with the same delivery.
number_of_assign_submission_grade = assign_submission_grade_cmi.count()
print(f"Number of assignment submissions with course grades {course_ip}: {number_of_assign_submission_grade}\n")

print("We check that there is no more than one submission per user for the same task.")

num_subByUser = (
    assign_submission_grade_cmi
    .groupBy("userid", "assignment")
    .count()
    .filter(col("count") > 1)
)

num_subByUser_count = num_subByUser.count()
print(f"Number of users with more than one submission per task: {num_subByUser_count} \n")
print("Name of the users with more than one delivery per task, along with the task for which they have more than one delivery and the number of deliveries")
num_subByUser.show(num_subByUser_count, truncate=False)

print("First five rows of the generated dataframe:")
assign_submission_grade_cmi.show(5, truncate=False)

assign_submission_grade_cmi.printSchema()

# # Write the dataframe to parquet
# assign_submission_grade_cmi.write.mode("overwrite").parquet(f"{destination_path}/assign_submission_grade_cmi.parquet")
# print("assign_submission_grade_cmi.parquet created :)")



Número de envíos de tareas del curso 8683: 1309
Número de envíos de tareas con notas del curso 8683: 1309

Comprobamos que no exista más de una entrega por usuario para una misma tarea.
Número de usuarios con más de una entrega por tarea: 4 

Nombre de los usuarios con más de una entrega por tarea, junto a la tarea en la que tienen más de una entrega y el número de entregas
+----------------------------------------------------------------+----------+-----+
|userid                                                          |assignment|count|
+----------------------------------------------------------------+----------+-----+
|5feceb66ffc86f38d952786c6d696c79c2dbc239dd4e91b46729d73a27fb57e9|108682    |29   |
|5feceb66ffc86f38d952786c6d696c79c2dbc239dd4e91b46729d73a27fb57e9|108640    |62   |
|5feceb66ffc86f38d952786c6d696c79c2dbc239dd4e91b46729d73a27fb57e9|109226    |61   |
|5feceb66ffc86f38d952786c6d696c79c2dbc239dd4e91b46729d73a27fb57e9|113721    |58   |
+----------------------------------

25/06/12 20:11:45 WARN MemoryManager: Total allocation exceeds 95,00% (1.020.054.720 bytes) of heap memory
Scaling row group sizes to 95,00% for 8 writers


As we can see, the number of tuples that the dataframe has resulting from the union of tasks with deliveries is equal to the value returned when calculating it for the dataframe resulting from the union of the previous one with the notes for each user and task. Therefore, it has been successfully verified that there is no more than one grade published for each assignment delivery.

On the other hand, it has been seen that, while in the majority of cases there is only a single delivery associated with a user, there are four users who present many more deliveries, something that is very strange.

## Data exploration

Next, the data obtained in the previous cell will be explored. Firstly, classrooms may have many activities published, but they remain hidden from students during the course since they are useful to teachers but are not intended to be carried out by students. Therefore, only those that have actually been published during the course should be considered for our metrics.

To do this, we will analyze, of the total number of tasks that are published in the classroom, which of them have at least one delivery registered, since this will mean that it was visible.

When calculating the union of the deliveries with the activities to give rise to the `submissions_ip` dataframe, an `inner join` was done; those tasks that did not have any deliveries associated with them did not meet the join condition and therefore were not returned in the operation. 

For this reason, to check which tasks really had a delivery, it will be enough for us to count the number of tuples in that dataframe, leaving only the values ​​other than the task identifier, to avoid counting the number of deliveries for each task.

In [3]:

from pyspark.sql.functions import from_unixtime, date_format

# Total number of course assignments
num_total_tareas_ip = assign_ip.select("id").distinct().count()

# Number of tasks with at least one associated delivery
num_tareas_con_entregas_ip = assign_submission_grade_cmi.select("assignment").distinct().count() 

print(f"Total number of tasks: {num_total_tareas_ip}")
print(f"Number of tasks with at least one delivery: {num_tareas_con_delgas_ip} \n")

print("========================================================\n")

print("Identifier, name and delivery date of all tasks")

assign_ip_f = assign_ip.withColumn(
      "duedate_formatted", date_format(from_unixtime(col("duedate")), "yy/MM/dd")
)
assign_ip_f.select("id", "name", "duedate_formatted").orderBy("name").show(100, truncate=False)

print("\n")
print("Identifier, name and delivery date of tasks with at least one delivery")

assign_submission_grade_cmi_F = assign_submission_grade_cmi.withColumn(
      "duedate_formatted", date_format(from_unixtime(col("duedate")), "yy/MM/dd")
)
assign_submission_grade_cmi_F.select(
      "assignment", "name", "duedate_formatted"
).dropDuplicates(["assignment"]).orderBy("name").show(100, truncate=False)
    

Número total de tareas: 34
Número de tareas con al menos una entrega: 14 


Identificador, nombre y fecha de entrega de todas las tareas
+------+--------------------------------------------------------------------------------------------------------------------------+-----------------+
|id    |name                                                                                                                      |duedate_formatted|
+------+--------------------------------------------------------------------------------------------------------------------------+-----------------+
|7592  |Actividad 00. Actualización del perfil en el campus virtual                                                               |23/09/25         |
|96123 |Calificación y comentarios de la actividad 02                                                                             |70/01/01         |
|107688|Entrega actividad 07                                                                                     

After looking closely at the exit, we can see something very strange.

The vast majority of classroom tasks do not have any associated delivery in the course being considered. It is necessary to study what this could be due to, given that it is very strange that there is only information on deliveries in the first assignment of the course, and on the deliveries of the subject project, but not on any of those that occur in the months of October and November. Furthermore, it can be seen that there are recorded deliveries of tasks that, in principle, were completed in previous years.

For example, let's look at all the submissions a random student made throughout a course.

In [4]:
as_sub_grad_submitFormatted = assign_submission_grade_cmi_F.withColumn(
    "timesubmitted_formatted", date_format(from_unixtime(col("timesubmitted")), "dd/MM/yy")
)
(as_sub_grad_submitFormatted.filter(col("userid") == "df7ce50a753bbb53b8e1d9eaf6a3432fa82b4e47e002142d2cf924451f3b39b1" )
      .select("userid", "assignment","name", "grade", "timesubmitted_formatted", "duedate_formatted")
      .orderBy("assignment", "timesubmitted_formatted")
      .show(100, truncate=False)
 )

+----------------------------------------------------------------+----------+-----------------------------------------------------------------------+--------+-----------------------+-----------------+
|userid                                                          |assignment|name                                                                   |grade   |timesubmitted_formatted|duedate_formatted|
+----------------------------------------------------------------+----------+-----------------------------------------------------------------------+--------+-----------------------+-----------------+
|df7ce50a753bbb53b8e1d9eaf6a3432fa82b4e47e002142d2cf924451f3b39b1|7592      |Actividad 00. Actualización del perfil en el campus virtual            |2.00000 |12/09/23               |23/09/25         |
|df7ce50a753bbb53b8e1d9eaf6a3432fa82b4e47e002142d2cf924451f3b39b1|107688    |Entrega actividad 07                                                   |-1.00000|19/11/23               |23/11/20      

## Union with users with student permission and enrolled in the subject

Next, we are going to check if there are changes in the data when we do a join with the table in which the ID of the students who were enrolled in the course last year is saved. Perhaps, in this way, users who had multiple deliveries for a single task are eliminated. Although logic tells us that having filtered the data of the tasks to only be left with that of the ip subject, and that therefore the data collected from the deliveries should coincide with the students of the subject, this way we ensure that no dirty data is leaked.

In [6]:
alumnos_ip = spark.read.parquet(f"{ruta_origen}/alumnos_ip_cmi.parquet")

print("\n")

# Join student data with deliveries and notes
as_sub_grad_alumnnos = assign_submission_grade_cmi.join(alumnos_ip, on="userid", how="inner")
print (f"Number of tuples in the table after joining with the enrolled students: {as_sub_grad_alumnnos.count()}\n")

print("We check if the user who had multiple deliveries on several tasks now exists")

as_sub_grad_alumnnos.filter(col("userid") == "5feceb66ffc86f38d952786c6d696c79c2dbc239dd4e91b46729d73a27fb57e9").select("userid").distinct().show() 

print("Now, we can check how many users have stopped being considered after joining with the enrolled students, that is, how many users who were present in the first table were really \n students of the subject that course\n")
num_users_1 = assign_submission_grade_cmi.select("userid").distinct().count()
num_users_2 = as_sub_grad_alumnnos.select("userid").distinct().count()
print(f"Number of users present in the first table without filtering by enrolled students: {num_users_1}\n")
print(f"Number of users present in the second table filtered by enrolled students: {num_users_2}\n")




Número de tuplas de la tabla tras unir con los alumnos matriculados: 1071

Comprobamos si existe ahora el usuario que tenía multiples entregas sobre varias tareas
+------+
|userid|
+------+
+------+

Ahora, podemos comprobar cuantos usuarios han dejado de ser considerados tras unir con los alumnos matriculados, es decir, cuantos usuarios de los que había presente en la primera tabla realmente 
 eran alumnos de la asignatura ese curso

Número de usuarios presentes en la primera tabla sin filtrar por alumnos matriculados : 209

Número de usuarios presentes en la segunda tabla filtrada por alumnos matriculados : 201



Firstly, we can see that, after performing the join, several tuples have disappeared, given that there were initially 1309 in the table at the beginning, and now several have disappeared, given that after the join there are 1071. Specifically, 8 users have disappeared from the table, and given that one of them we have been able to verify was the user who had multiple deliveries on several tasks, it makes sense that the number of tuples in the table has decreased after these users disappeared.

Furthermore, we can observe that the user who had multiple deliveries in several of the tasks has disappeared when joining only with the enrolled students, perhaps because he has disappeared when filtering by students, because some detail related to the teachers' interactions with the deliveries was also saved.

Finally, we can check if the number of tasks that have deliveries remains constant after this union, or if it has changed and some have disappeared.

In [7]:
print(f"Number of tasks with at least one delivery prior to joining with the enrolled students: {num_tareas_con_deliveries_ip}")
print(f"Number of assignments with at least one submission after joining with enrolled students: {as_sub_grad_alumnnos.select('assignment').distinct().count()}\n")
print("========================================================\n")
print("Identifier, name and delivery date of assignments with at least one delivery before joining with enrolled students")
assign_submission_grade_cmi_F.select(
      "assignment", "name", "duedate_formatted"
).dropDuplicates(["assignment"]).orderBy("name").show(100, truncate=False)
print("\n")
print("Identifier, name and delivery date of the tasks with at least one delivery after joining with the enrolled students")
as_sub_grad_alumnnos_F = as_sub_grad_alumnnos.withColumn(
      "duedate_formatted", date_format(from_unixtime(col("duedate")), "dd/MM/yy")
)
as_sub_grad_alumnnos_F.select(
      "assignment", "name", "duedate_formatted"
).dropDuplicates(["assignment"]).orderBy("name").show(100, truncate=False)
print("\n")




Número de tareas con al menos una entrega previamente a la unión con los alumnos matriculados: 14
Número de tareas con al menos una entrega tras la unión con los alumnos matriculados: 10


Identificador, nombre y fecha de entrega de las tareas con al menos una entrega antes de unir con los alumnos matriculados
+----------+-----------------------------------------------------------------------+-----------------+
|assignment|name                                                                   |duedate_formatted|
+----------+-----------------------------------------------------------------------+-----------------+
|7592      |Actividad 00. Actualización del perfil en el campus virtual            |23/09/25         |
|96123     |Calificación y comentarios de la actividad 02                          |70/01/01         |
|107688    |Entrega actividad 07                                                   |23/11/20         |
|108682    |Entrega de la actividad 08 - Proyecto - Parejas - NO VALE 

Finally, you can see how those tasks that were from other years but that were surely delivered in extraordinary calls by students from previous years have disappeared from the dataframe after the union with the students, given that after filtering by only those who were enrolled in the virtual classroom that year, we only have information on tasks whose delivery date coincides with the months in which the course is taught.

However, it should be noted that there is still no trace of any assignments completed or delivered in October, and hardly any in November, which in principle are very active months of assignments in that course.

## Task verification in October

Lastly, we are going to make sure that there are no assignments actually in the classroom in the month of October, nor any submissions on that date

In [11]:
from pyspark.sql.functions import to_date
from pyspark.sql.functions import to_timestamp


# Convert 'duedate_formatted' column to date type
assign_ip_f = assign_ip_f.withColumn(
    "duedate_date", to_date(col("duedate_formatted"), "yy/MM/dd")
)

print("Let's explore how many IP tasks have a delivery date set for sometime in October\n")
filtered_assignments = assign_ip_f.filter(
    (col("duedate_date") > "2023-10-01") & (col("duedate_date") < "2023-10-31")
)

filtered_assignments.select("id", "name", "duedate_date").show(truncate=False)

print(
    "Ahora vamos a comprobar si existen entregas que hayan sido entregadas durante el mes de octubre de 2023\n"
)


# Convert 'timesubmitted' column to timestamp type
sub_ip_date = submissions_ip.withColumn(
    "timesubmitted_date", to_timestamp(from_unixtime(col("timesubmitted")), "yyyy-MM-dd HH:mm:ss")
)

# Filter deliveries whose date is after '11/01/23' and before '11/30/23'
filtered_submissions = sub_ip_date.filter(
    (col("timesubmitted_date") > "2023-10-01 00:00:00") &
    (col("timesubmitted_date") < "2023-10-31 23:59:59")
)

filtered_submissions.select("userid", "assignment", "timesubmitted_date", "name").drop_duplicates(["assignment"]).show(100, truncate=False)

Vamos a explorar cuantas tareas de ip tienen fijada la fecha de entrega en algún momento de octubre

+---+----+------------+
|id |name|duedate_date|
+---+----+------------+
+---+----+------------+

Ahora vamos a comprobar si existen entregas que hayan sido entregadas durante el mes de octubre de 2023



+----------------------------------------------------------------+----------+-------------------+-----------------------------------------------------------+
|userid                                                          |assignment|timesubmitted_date |name                                                       |
+----------------------------------------------------------------+----------+-------------------+-----------------------------------------------------------+
|091af124e119a447c7f6594fb2f7c4fbb678f669966db01e3f62c26eedb220af|7592      |2023-10-01 19:46:08|Actividad 00. Actualización del perfil en el campus virtual|
+----------------------------------------------------------------+----------+-------------------+-----------------------------------------------------------+



It can be seen that there are only deliveries of tasks `7592` in October, this being task only activity 0.

# Writing

In [14]:
as_sub_grad_alumnnos.printSchema()

as_sub_grad_alumnnos.orderBy("userid").show(200, truncate=False)

as_sub_grad_alumnnos.groupBy("userid", "assignment").count().orderBy("userid").show(200, truncate=False)

as_sub_grad_alumnnos.write.mode("overwrite").parquet(f"{ruta_destino}/assign_sub_grad_al_ip.parquet")

root
 |-- userid: string (nullable = true)
 |-- assignment: long (nullable = true)
 |-- timesubmitted: long (nullable = true)
 |-- duedate: long (nullable = true)
 |-- allowsubmissionsfromdate: long (nullable = true)
 |-- name: string (nullable = true)
 |-- grade: string (nullable = true)

+----------------------------------------------------------------+----------+-------------+----------+------------------------+-----------------------------------------------------------------------+--------+
|userid                                                          |assignment|timesubmitted|duedate   |allowsubmissionsfromdate|name                                                                   |grade   |
+----------------------------------------------------------------+----------+-------------+----------+------------------------+-----------------------------------------------------------------------+--------+
|006b0e7bd07cec05e0952cb61c30893f6d30d7962f9efc99d0f041f6fadcc320|7592      |16971